# Skin Condition Classifier — Phase 2: Data Preprocessing
This notebook downloads both datasets from Kaggle, explores them, selects the best classes, and prepares everything for model training in Phase 3.

**Run each cell from top to bottom. Do not skip any cells.**

## Cell 1 — Upload your Kaggle API key
This will open a file picker. Navigate to your project folder and select `kaggle.json`.

In [ ]:
from google.colab import files

print('Please upload your kaggle.json file...')
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API key configured successfully.')

## Cell 2 — Install Kaggle and download both datasets
This downloads ACNE04 and DermNet directly from Kaggle into Colab's temporary storage. No Google Drive space used.

In [ ]:
!pip install kaggle --quiet

os.makedirs('/content/data/acne04', exist_ok=True)
os.makedirs('/content/data/dermnet', exist_ok=True)

print('Downloading ACNE04 dataset...')
!kaggle datasets download -d jincyjis/acne04 -p /content/data/acne04 --unzip
print('ACNE04 downloaded.')

print('Downloading DermNet dataset...')
!kaggle datasets download -d shubhamgoel27/dermnet -p /content/data/dermnet --unzip
print('DermNet downloaded.')

print('Both datasets ready.')

## Cell 3 — Import libraries
Loading everything we need for data exploration, processing, and visualization.

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image
from collections import Counter

print('Libraries loaded.')

## Cell 4 — Explore ACNE04 dataset
Let's see exactly how many images are in each severity level.

In [ ]:
acne_root = Path('/content/data/acne04')

print('ACNE04 folder structure:')
for item in sorted(acne_root.rglob('*')):
    if item.is_dir():
        imgs = list(item.glob('*.jpg')) + list(item.glob('*.png'))
        if imgs:
            print(f'  {item.name}: {len(imgs)} images')

# Count by severity level
acne_counts = {}
severity_map = {
    'levle0': 'acne_clear',
    'levle1': 'acne_mild',
    'levle2': 'acne_moderate',
    'levle3': 'acne_severe'
}

for folder in sorted(acne_root.rglob('*')):
    if folder.is_dir():
        for prefix, label in severity_map.items():
            if folder.name.startswith(prefix) or folder.name == prefix:
                imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
                acne_counts[label] = acne_counts.get(label, 0) + len(imgs)

print('\nACNE04 class counts:')
for label, count in sorted(acne_counts.items()):
    print(f'  {label}: {count} images')

## Cell 5 — Explore DermNet dataset
See all available classes and their image counts.

In [ ]:
dermnet_train = Path('/content/data/dermnet/train')

print('DermNet classes (train set):')
dermnet_counts = {}
for folder in sorted(dermnet_train.iterdir()):
    if folder.is_dir():
        imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
        dermnet_counts[folder.name] = len(imgs)
        print(f'  {folder.name}: {len(imgs)} images')

print(f'\nTotal classes: {len(dermnet_counts)}')
print(f'Total images: {sum(dermnet_counts.values())}')

## Cell 6 — Select DermNet classes to use
We are selecting the most relevant everyday skin conditions and excluding cancer-related or overly niche classes. These pair naturally with ACNE04.

In [ ]:
# These are the DermNet classes we want to keep.
# We exclude skin cancers (Melanoma, BCC) since this app focuses on everyday conditions.
SELECTED_CLASSES = [
    'Acne and Rosacea Photos',
    'Atopic Dermatitis Photos',
    'Eczema Photos',
    'Psoriasis pictures Lichen Planus and related diseases',
    'Seborrheic Keratoses and other Benign Tumors',
    'Nail Fungus and other Nail Disease',
    'Urticaria Hives',
    'Warts Molluscum and other Viral Infections',
]

# Clean label names for display in the app
CLASS_LABEL_MAP = {
    'Acne and Rosacea Photos': 'acne_rosacea',
    'Atopic Dermatitis Photos': 'atopic_dermatitis',
    'Eczema Photos': 'eczema',
    'Psoriasis pictures Lichen Planus and related diseases': 'psoriasis',
    'Seborrheic Keratoses and other Benign Tumors': 'seborrheic_keratoses',
    'Nail Fungus and other Nail Disease': 'nail_fungus',
    'Urticaria Hives': 'urticaria',
    'Warts Molluscum and other Viral Infections': 'warts',
}

print('Selected classes and image counts:')
for cls in SELECTED_CLASSES:
    count = dermnet_counts.get(cls, 0)
    label = CLASS_LABEL_MAP[cls]
    print(f'  {label}: {count} images')

## Cell 7 — Visualize sample images from each dataset
Always look at your data before training. This helps catch corrupt images, wrong labels, or quality issues.

In [ ]:
def show_samples(folder, title, n=4):
    imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    imgs = random.sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, imgs):
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.axis('off')
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

# Show ACNE04 samples
for folder in sorted(acne_root.rglob('*')):
    if folder.is_dir():
        imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
        if len(imgs) > 0:
            show_samples(folder, f'ACNE04: {folder.name}')
            break  # Just show one level as a sample

# Show DermNet samples for selected classes
for cls in SELECTED_CLASSES[:3]:
    folder = dermnet_train / cls
    if folder.exists():
        show_samples(folder, f'DermNet: {CLASS_LABEL_MAP[cls]}')

## Cell 8 — Build unified dataset folder
We create a single clean folder called `processed/` with one subfolder per class. ACNE04 severity levels and selected DermNet classes all live side by side here.

In [ ]:
PROCESSED_DIR = Path('/content/data/processed')
IMG_SIZE = (224, 224)  # MobileNetV2 standard input size
MAX_PER_CLASS = 1000   # Cap each class at 1000 images to keep things balanced

def process_and_save(src_paths, dest_folder, max_images=MAX_PER_CLASS):
    dest_folder.mkdir(parents=True, exist_ok=True)
    paths = random.sample(src_paths, min(max_images, len(src_paths)))
    saved = 0
    for i, path in enumerate(paths):
        try:
            img = Image.open(path).convert('RGB')
            img = img.resize(IMG_SIZE, Image.LANCZOS)
            img.save(dest_folder / f'{dest_folder.name}_{i:04d}.jpg', 'JPEG', quality=90)
            saved += 1
        except Exception as e:
            pass  # Skip corrupt images silently
    return saved

print('Building unified processed dataset...')
summary = {}

# --- Process ACNE04 ---
for prefix, label in severity_map.items():
    all_imgs = []
    for folder in sorted(acne_root.rglob('*')):
        if folder.is_dir() and (folder.name.startswith(prefix) or folder.name == prefix):
            all_imgs += list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
    if all_imgs:
        dest = PROCESSED_DIR / label
        n = process_and_save(all_imgs, dest)
        summary[label] = n
        print(f'  {label}: {n} images saved')

# --- Process DermNet selected classes ---
for cls in SELECTED_CLASSES:
    label = CLASS_LABEL_MAP[cls]
    all_imgs = []
    for split in ['train', 'test']:
        folder = Path(f'/content/data/dermnet/{split}') / cls
        if folder.exists():
            all_imgs += list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    if all_imgs:
        dest = PROCESSED_DIR / label
        n = process_and_save(all_imgs, dest)
        summary[label] = n
        print(f'  {label}: {n} images saved')

print(f'\nDone. Total classes: {len(summary)}')
print(f'Total images: {sum(summary.values())}')

## Cell 9 — Visualize class distribution
Check that no single class dominates the dataset. Balanced classes = fairer model.

In [ ]:
labels = list(summary.keys())
counts = list(summary.values())

plt.figure(figsize=(12, 5))
bars = plt.bar(labels, counts, color='steelblue', edgecolor='white')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.ylabel('Number of images')
plt.title('Images per class after processing')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(count), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('Class distribution looks good if bars are roughly similar height.')
print('Any class below 200 images may underperform — we will address this in Phase 3 with augmentation.')

## Cell 10 — Train / Validation / Test split
We split each class into 70% train, 15% validation, 15% test. This is done per class so every split is balanced.

In [ ]:
SPLIT_DIR = Path('/content/data/split')
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# Test gets the remaining 15%

random.seed(42)  # For reproducibility

split_summary = {'train': {}, 'val': {}, 'test': {}}

for class_folder in sorted(PROCESSED_DIR.iterdir()):
    if not class_folder.is_dir():
        continue
    class_name = class_folder.name
    all_imgs = list(class_folder.glob('*.jpg'))
    random.shuffle(all_imgs)

    n = len(all_imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = {
        'train': all_imgs[:n_train],
        'val':   all_imgs[n_train:n_train + n_val],
        'test':  all_imgs[n_train + n_val:]
    }

    for split_name, split_imgs in splits.items():
        dest = SPLIT_DIR / split_name / class_name
        dest.mkdir(parents=True, exist_ok=True)
        for img_path in split_imgs:
            shutil.copy(img_path, dest / img_path.name)
        split_summary[split_name][class_name] = len(split_imgs)

print('Train / Val / Test split complete:')
for split_name, classes in split_summary.items():
    total = sum(classes.values())
    print(f'  {split_name}: {total} images across {len(classes)} classes')

## Cell 11 — Save class names to a file
This saves the list of class names so the model and app always use the exact same labels.

In [ ]:
class_names = sorted([f.name for f in (SPLIT_DIR / 'train').iterdir() if f.is_dir()])

with open('/content/data/class_names.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print('Class names saved to class_names.txt:')
for i, name in enumerate(class_names):
    print(f'  {i}: {name}')

## Cell 12 — Download processed data and class names
Download `class_names.txt` to your local machine and save it into your project's `model/` folder. You will need it in Phase 3 and Phase 5.

The full split dataset stays in Colab for now — we will use it directly in the Phase 3 training notebook.

In [ ]:
from google.colab import files
files.download('/content/data/class_names.txt')
print('class_names.txt downloaded. Save it to your local model/ folder.')

## Phase 2 complete
Your data is cleaned, resized to 224x224, split into train/val/test, and ready for model training.

**Next step:** Open the Phase 3 notebook to train MobileNetV2 on this data.

**Important:** Do not close this Colab session until you have finished Phase 3 training — the processed data lives in Colab's temporary storage and will be lost if the session ends.